## Data (not in git)
- Podcast: NPR Planet Money, Episode 1,000 (2020)
- Transcript: https://www.npr.org/transcripts/857106873
- Put the files in `samples/` (this folder is ignored by git):
  - `samples/planet_money_1000.mp3`: the episode audio from the podcast feed
  - `samples/transcript_planet_money_1000.txt`: the transcript text, from "BYLINE: This is PLANET MONEY from NPR." to the last line
- The podcast feed version has ads that are not in the transcript (one at the start, three later). The notebook skips the first 25 seconds and removes the other ads before WER.

In [46]:
from faster_whisper import WhisperModel
from pathlib import Path

from faster_whisper import decode_audio
from IPython.display import Audio

In [47]:
model = WhisperModel("small", device="cpu", compute_type="int8", cpu_threads=8)

In [48]:
SAMPLES = Path("../samples")    
SAMPLE_RATE = 16000              # Whisper expects 16 kHz !!

In [49]:
audio = decode_audio(str(SAMPLES / "planet_money_1000.mp3")) # https://www.npr.org/transcripts/857106873
print(f"{len(audio) / SAMPLE_RATE / 60:.1f} minutes")


27.7 minutes


In [50]:
Audio(audio[: 30 * SAMPLE_RATE], rate=SAMPLE_RATE)    # listen to the first 30 seconds

In [51]:
transcript = (SAMPLES / "transcript_planet_money_1000.txt").read_text(encoding="utf-8")

print(len(transcript.split()), "words")
print(transcript[:500])        # the first 500 characters

4913 words
UNIDENTIFIED PERSON #1, BYLINE: This is PLANET MONEY from NPR.

(SOUNDBITE OF SONG, "PLANET MONEY (IT'S THEIR 1,000TH EPISODE)")

SCOTT LANE AND DEAU EYES: (Singing) PLANET MONEY - it's their 1,000th episode.

KENNY MALONE, HOST:

Yeah, it's our 1,000th episode. And to be honest, we spent a lot of time thinking about ways to avoid doing this episode.

SARAH GONZALEZ, HOST:

There's a lot going on right now. So at one point, we were like, what if we just skip episode 1,000, go from Episode 999 to


In [52]:
help(model)

Help on WhisperModel in module faster_whisper.transcribe object:

class WhisperModel(builtins.object)
 |  WhisperModel(model_size_or_path: str, device: str = 'auto', device_index: Union[int, List[int]] = 0, compute_type: str = 'default', cpu_threads: int = 0, num_workers: int = 1, download_root: Optional[str] = None, local_files_only: bool = False, files: dict = None, revision: Optional[str] = None, use_auth_token: Union[str, bool, NoneType] = None, **model_kwargs)
 |
 |  Methods defined here:
 |
 |  __init__(self, model_size_or_path: str, device: str = 'auto', device_index: Union[int, List[int]] = 0, compute_type: str = 'default', cpu_threads: int = 0, num_workers: int = 1, download_root: Optional[str] = None, local_files_only: bool = False, files: dict = None, revision: Optional[str] = None, use_auth_token: Union[str, bool, NoneType] = None, **model_kwargs)
 |      Initializes the Whisper model.
 |
 |      Args:
 |        model_size_or_path: Size of the model to use (tiny, tiny.en, b

In [53]:
clip = audio[: 5 * 60 * SAMPLE_RATE]
text_1=model.transcribe(clip, task="transcribe")

type(text_1)
text_1

(<generator object WhisperModel.generate_segments at 0x000001DA138A86C0>,
 TranscriptionInfo(language='en', language_probability=0.9935786724090576, duration=300.0, duration_after_vad=300.0, all_language_probs=[('en', 0.9935786724090576), ('es', 0.0012867189943790436), ('ru', 0.0007215774967335165), ('de', 0.0006259651272557676), ('fr', 0.0004657450772356242), ('la', 0.00044709607027471066), ('ko', 0.0003391726640984416), ('pt', 0.0002677953743841499), ('ar', 0.00026479767984710634), ('ja', 0.0001925016986206174), ('nn', 0.00017956893134396523), ('id', 0.00015728920698165894), ('zh', 0.00014653279504273087), ('it', 0.00011434143380029127), ('cy', 0.00010892518912442029), ('haw', 0.00010631074837874621), ('nl', 9.059525473276153e-05), ('sn', 7.583469414385036e-05), ('vi', 7.181349792517722e-05), ('km', 6.668407877441496e-05), ('jw', 5.54108846699819e-05), ('th', 5.214936027186923e-05), ('pl', 4.8772279114928097e-05), ('el', 4.576927312882617e-05), ('ms', 4.122479003854096e-05), ('mi', 3

## Database integration

### Connect and validate

In [54]:
import sqlite3
from pathlib import Path

DB_PATH = "../database/app.db"   # created by running database.ipynb
REQUIRED_TABLES = {"sources", "transcript_segments", "claims", "claim_segments",
                   "verified_claims", "evidence", "verdict_evidence"}


def check_database(db_path: str = DB_PATH) -> None:
    """Stop with a clear message unless the database file exists and has the schema."""
    if not Path(db_path).exists():
        raise FileNotFoundError(f"{db_path} does not exist. Run database.ipynb first.")
    con = sqlite3.connect(db_path)
    try:
        tables = {r[0] for r in con.execute(
            "SELECT name FROM sqlite_master WHERE type = 'table'")}
    finally:
        con.close()
    missing = REQUIRED_TABLES - tables
    if missing:
        raise RuntimeError(f"{db_path} is missing tables {sorted(missing)}. "
                           "Run database.ipynb first.")
    print(f"database OK: {db_path}")


check_database()

database OK: ../database/app.db


### Database helpers

In [55]:
WINDOW_S = 30      # length of each window
OVERLAP_S = 5      # audio shared with the next window

In [56]:
def transcribe_and_save(audio, source_url, window_s=WINDOW_S, overlap_s=OVERLAP_S,
                        db_path=DB_PATH):
    """Transcribe in overlapping windows, committing each window's rows as it finishes.
    Returns the new source id."""
    check_database(db_path)
    con = sqlite3.connect(db_path, timeout=5)
    con.execute("PRAGMA foreign_keys = ON")
    step_s = window_s - overlap_s
    total_s = len(audio) / SAMPLE_RATE
    try:
        with con:  # create the source row first, so the DB shows it straight away
            cur = con.execute(
                "INSERT INTO sources (source_url, status) VALUES (?, 'processing')",
                (source_url,),
            )
        source_id = cur.lastrowid

        start_s = 0.0
        while True:
            end_s = min(start_s + window_s, total_s)
            clip = audio[int(start_s * SAMPLE_RATE) : int(end_s * SAMPLE_RATE)]
            segments, _ = model.transcribe(clip, language="en", task="transcribe",
                                           beam_size=5, vad_filter=True)
            is_last = end_s >= total_s
            owned_end = total_s if is_last else start_s + step_s   # this window's share
            rows = [
                (source_id, round((start_s + s.start) * 1000),
                 round((start_s + s.end) * 1000), s.text.strip())
                for s in segments
                if start_s + s.start < owned_end   # the next window owns the overlap
            ]
            with con:  # commit this window now, so readers see it while the loop runs
                con.executemany(
                    """INSERT INTO transcript_segments
                           (video_id, start_ms, end_ms, text, is_stable)
                       VALUES (?, ?, ?, ?, 1)""",
                    rows,
                )
            print(f"window {start_s:7.1f}-{end_s:7.1f} s: {len(rows)} segments saved")

            if is_last:
                break
            start_s += step_s

        with con:
            con.execute("UPDATE sources SET status = 'done' WHERE id = ?", (source_id,))
        return source_id
    finally:
        con.close()

## Transcription loop

In [57]:
import re

import jiwer
from transformers import WhisperTokenizer

reference = transcript
reference = re.sub(r"^\(.*\)\s*$", "", reference, flags=re.M)             # lines that are only a sound note, e.g. (SOUNDBITE OF MUSIC)
reference = re.sub(r"^[A-Z][A-Z0-9 .,'#&-]*:", "", reference, flags=re.M)  # speaker names, e.g. KENNY MALONE, HOST:

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-large-v3-turbo")
ref_norm = tokenizer.normalize(reference)

print(len(ref_norm.split()), "words")      # should be 4435
print(ref_norm[:300])

4435 words
this is planet money from npr planet money it is their 1000th episode yeah it is our 1000th episode and to be honest we spent a lot of time thinking about ways to avoid doing this episode there is a lot going on right now so at one point we were like what if we just skip episode 1000 go from episode


In [58]:
import time

t0 = time.perf_counter()
source_id = transcribe_and_save(audio, "planet_money_1000.mp3")
elapsed = time.perf_counter() - t0

con = sqlite3.connect(DB_PATH)
try:
    rows = con.execute(
        "SELECT start_ms, text FROM transcript_segments WHERE video_id = ? ORDER BY start_ms",
        (source_id,),
    ).fetchall()
finally:
    con.close()

for start_ms, text in rows:
    print(f"[{start_ms / 1000:7.1f}] {text}")

hypothesis = " ".join(text for _, text in rows)   # read back from the DB for the WER cell
audio_s = len(audio) / SAMPLE_RATE
print(f"\nsaved source {source_id}: {len(rows)} segments")
print(f"audio {audio_s:.0f}s  took {elapsed:.0f}s  RTF {elapsed / audio_s:.2f}")


database OK: ../database/app.db
window     0.0-   30.0 s: 5 segments saved
window    25.0-   55.0 s: 6 segments saved
window    50.0-   80.0 s: 12 segments saved
window    75.0-  105.0 s: 8 segments saved
window   100.0-  130.0 s: 13 segments saved
window   125.0-  155.0 s: 7 segments saved
window   150.0-  180.0 s: 12 segments saved
window   175.0-  205.0 s: 7 segments saved
window   200.0-  230.0 s: 13 segments saved
window   225.0-  255.0 s: 6 segments saved
window   250.0-  280.0 s: 5 segments saved
window   275.0-  305.0 s: 7 segments saved
window   300.0-  330.0 s: 10 segments saved
window   325.0-  355.0 s: 7 segments saved
window   350.0-  380.0 s: 8 segments saved
window   375.0-  405.0 s: 12 segments saved
window   400.0-  430.0 s: 12 segments saved
window   425.0-  455.0 s: 8 segments saved
window   450.0-  480.0 s: 7 segments saved
window   475.0-  505.0 s: 10 segments saved
window   500.0-  530.0 s: 4 segments saved
window   525.0-  555.0 s: 6 segments saved
window   550.0

In [59]:
hyp_norm = tokenizer.normalize(hypothesis)

out = jiwer.process_words(ref_norm, hyp_norm)
print(f"WER {100 * out.wer:.1f} %")
print(f"wrong {out.substitutions}, missing {out.deletions}, extra {out.insertions} "
      f"(reference: {len(ref_norm.split())} words)")

WER 24.5 %
wrong 171, missing 83, extra 834 (reference: 4435 words)


In [60]:
for chunk in out.alignments[0]:
    if chunk.type == "insert":
        n = chunk.hyp_end_idx - chunk.hyp_start_idx
        if n >= 10:
            words = out.hypotheses[0][chunk.hyp_start_idx : chunk.hyp_end_idx]
            print(f"{n} extra words: {' '.join(words[:40])} ...\n")

28 extra words: week on up 1st the midterms less than 15 days away president trump vice president j d vance and former president obama all crisscrossing the country rallying voters ...

26 extra words: michigan to georgia to north carolina join us every morning as we guide you through the home stretch of this election season up 1st on the ...

10 extra words: i am welcome to planet money welcome to planet money ...

209 extra words: election season feels messier than ever attack ads misinformation ai npr is your antidote our network of real human journalists keeps you focused on the facts and with npr plus you directly support journalists on the ground asking questions in ...

13 extra words: part of the episode after the break this is what we call the ...

14 extra words: context but also to be talking to the people in the middle who are ...

16 extra words: that real example which i know does not sound enticing but back in 2011 the case ...

158 extra words: american journalist austi

In [61]:
# the three long "extra" blocks printed above are ads, not part of the episode
ads = [c for c in out.alignments[0]
       if c.type == "insert" and c.hyp_end_idx - c.hyp_start_idx >= 40]

hyp_words = out.hypotheses[0]
kept = [w for i, w in enumerate(hyp_words)
        if not any(c.hyp_start_idx <= i < c.hyp_end_idx for c in ads)]

out_clean = jiwer.process_words(ref_norm, " ".join(kept))
print(f"removed {len(hyp_words) - len(kept)} ad words")
print(f"WER {100 * out_clean.wer:.1f} %")
print(f"wrong {out_clean.substitutions}, missing {out_clean.deletions}, extra {out_clean.insertions}")

removed 418 ad words
WER 15.1 %
wrong 171, missing 83, extra 416
